# LoRA FineTuned Qwen RAG FactCheck System

## 1. Configuration and Argument Setup
Sets Python variables for paths to raw evidence (evidence.json), the SQLite database, the base model ID (Qwen/Qwen3.5-2B), FAISS index paths, and the fine-tuned LoRA weights path.

In [ ]:
# Args
EVIDENCE_JSON_PATH = 'evidence.json'
EVIDENCE_DB_PATH = 'evidence.db'
MODEL_ID = "Qwen/Qwen3.5-2B"
COT_DB_PATH = "cot.db"
RAG_MODEL_NAME = 'all-MiniLM-L6-v2'
FAISS_INDEX_PATH = 'faiss_index.bin'
FAISS_META_PATH = 'faiss_metadata.json'
LORA_PATH = "model/qwen-cot-lora-final"
DEV_FILE_PATH = "dev-claims.json"

## 2. Environment Installation
Installs all necessary Python libraries required to run the notebook

In [ ]:
#Run this and restart seesion
!pip install ijson
!pip install --upgrade transformers
!pip install -U bitsandbytes>=0.46.1
!pip install trl
!pip install faiss-gpu-cu12
!pip install sentence_transformers

## 3. Building the Evidence Database
Converts the large raw JSON evidence file into a structured SQLite database for efficient ID-based retrieval.

In [ ]:
import sqlite3
import ijson
import os

def build_evidence_db(json_path, db_path):
    if os.path.exists(db_path):
        print(f"Database {db_path} already exists. Skipping build.")
        return

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS evidence (id TEXT PRIMARY KEY, text TEXT)')

    print("Building database... This may take a few minutes.")

    with open(json_path, 'r', encoding='utf-8') as f:
        items = ijson.kvitems(f, '')

        batch = []
        for count, (ev_id, ev_text) in enumerate(items):
            batch.append((ev_id, ev_text))

            if len(batch) >= 1000:
                cursor.executemany('INSERT OR IGNORE INTO evidence (id, text) VALUES (?, ?)', batch)
                batch = []
                if count % 10000 == 0:
                    print(f"Processed {count} items...")

        if batch:
            cursor.executemany('INSERT OR IGNORE INTO evidence (id, text) VALUES (?, ?)', batch)

    conn.commit()
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_id ON evidence (id)')
    conn.close()
    print("Database build complete!")

build_evidence_db(EVIDENCE_JSON_PATH, EVIDENCE_DB_PATH)

## 4. Evidence Retrieval Utility
Provides a helper function to fetch specific evidence text using its ID.

In [ ]:
import sqlite3
import os

def get_evidence_text(evidence_id, db_path=EVIDENCE_DB_PATH):
    if not os.path.exists(db_path):
        script_dir = os.path.dirname(os.path.abspath(__file__))
        potential_db_path = os.path.join(script_dir, 'evidence.db')
        if os.path.exists(potential_db_path):
            db_path = potential_db_path
        else:
            print(f"Error: Database file not found at {db_path}")
            return None

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        cursor.execute('SELECT text FROM evidence WHERE id = ?', (evidence_id,))
        result = cursor.fetchone()

        conn.close()

        if result:
            return result[0]
        else:
            return None

    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None

## 5. Chain of Thought (CoT) Generation
Uses the base Qwen model to generate logical reasoning (CoT) for the training set, which will later be used as training data for fine-tuning.
Implementation:


*   Loads the Qwen model in 4-bit quantization to save VRAM.
*   Uses a few-shot prompt template that instructs the model to "analyze step by step" before providing a conclusion.
*   Iterates through the training claims and saves the generated reasoning into a new database (cot.db).
*   Supports check-pointing by skipping claims already present in the database.


In [ ]:
import json
import sqlite3
import torch
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

try:
    # pyrefly: ignore [missing-import]
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except (ImportError, ModuleNotFoundError):
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

login(token=hf_token)

model_id = "Qwen/Qwen3.5-2B"

# Left-padding is required for batched generation so all sequences
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=quantization_config,
    trust_remote_code=True
)

model.generation_config.max_length = None
model.config.pad_token_id = tokenizer.pad_token_id

# ── DB ────────────────────────────────────────────────────────────────────────

def init_db(db_path=COT_DB_PATH):
    dir_name = os.path.dirname(db_path)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS reasoning_data (
            claim_id TEXT PRIMARY KEY,
            claim_text TEXT,
            label TEXT,
            reasoning TEXT
        )
    ''')
    conn.commit()
    return conn

# ── Prompt ────────────────────────────────────────────────────────────────────

def format_prompt(evidence, claim, label):
    few_shot = """You are a climate science fact-checker. Your task is to explain the logical connection between the Evidence and the Claim to justify the Label. Use the format: "Let's analyze step by step: [Reasoning] Therefore, the conclusion is: [Label]."
                    Evidence: 1. CO2 can be toxic to animals at 10,000 ppm. 2. Plants grow faster at 1,000 ppm CO2. 3. Higher CO2 affects plant growth favorably.
                    Claim: Higher CO2 concentrations actually help ecosystems support more plant and animal life.
                    Label: DISPUTED
                    Let's analyze step by step: The evidence confirms that CO2 promotes plant growth, which supports part of the claim. However, it also notes that extremely high concentrations are toxic to animal life. Since the claim makes a broad positive statement without accounting for these toxic thresholds, the claim is partially accurate but also potentially dangerous/misleading.
                    Therefore, the conclusion is: DISPUTED.

                    Evidence: 1. Human activity and GHG emissions are key factors in global temperature increases. 2. Warming is driven by human-caused thermal expansion and melting ice.
                    Claim: El Niño drove record highs in global temperatures suggesting rise may not be down to man-made emissions.
                    Label: REFUTES
                    Let's analyze step by step: While El Niño is a natural driver of temperature, the evidence explicitly states that human activity is the "key factor" in the pace of current temperature increases. The claim attempts to dismiss man-made emissions by pointing to a natural cause, which contradicts the "substantial evidence" mentioned in the text regarding human-caused warming.
                    Therefore, the conclusion is: REFUTES.

                    Evidence: 1. Reversals in polarity occurred around 1925, 1947, and 1977. 2. The PDO changed to a "cool" phase in a regime shift similar to the 1970s.
                    Claim: In 1946, PDO switched to a cool phase.
                    Label: SUPPORTS
                    Let's analyze step by step: The evidence mentions a major PDO reversal occurring around 1947 and explicitly describes a shift to a "cool" phase. The year 1946 is immediately adjacent to the 1947 reversal date cited. Given the context of regime shifts, the evidence provides sufficient support for the timing and nature of the phase change described in the claim.
                    Therefore, the conclusion is: SUPPORTS.

                    Evidence: {evidence}
                    Claim: {claim}
                    Label: {label}
                    Let's analyze step by step:"""
    return few_shot.format(evidence=evidence, claim=claim, label=label)

# ── Batched generation ────────────────────────────────────────────────────────

def generate_batch(batch_items: list[dict], max_new_tokens: int = 300) -> list[str]:
    """
    Tokenize each prompt individually (as-is), then left-pad into a single
    batch tensor and call model.generate() once for the whole batch.

    Returns a list of decoded reasoning strings, one per item.
    """
    prompts = [item["prompt"] for item in batch_items]

    # Single-item tokenization (no padding here, just get token ids)
    input_ids_list = [
        tokenizer(p, return_tensors="pt")["input_ids"][0]
        for p in prompts
    ]

    # Manual left-pad so all sequences end at the same position
    max_len = max(ids.shape[0] for ids in input_ids_list)
    padded_ids = []
    attention_masks = []
    for ids in input_ids_list:
        pad_len = max_len - ids.shape[0]
        padded = torch.cat([
            torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long),
            ids
        ])
        mask = torch.cat([
            torch.zeros(pad_len, dtype=torch.long),
            torch.ones(ids.shape[0], dtype=torch.long)
        ])
        padded_ids.append(padded)
        attention_masks.append(mask)

    input_ids = torch.stack(padded_ids).to(model.device)
    attention_mask = torch.stack(attention_masks).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    results = []
    for out in outputs:
        new_tokens = out[max_len:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        results.append(text)

    return results

# ── Main ──────────────────────────────────────────────────────────────────────

BATCH_SIZE = 12
MAX_NEW_TOKENS = 300

db_conn = init_db()
cursor = db_conn.cursor()

# Checkpoint / resume: skip already-processed claims
cursor.execute("SELECT claim_id FROM reasoning_data")
done_ids = {row[0] for row in cursor.fetchall()}
print(f"Resuming: {len(done_ids)} claims already in DB.")

with open('train-claims.json', 'r') as f:
    train_data = json.load(f)

all_claims_ready = []
for cid, data in train_data.items():
    if cid in done_ids:
        continue

    evidence_pieces = []
    for i, ev_id in enumerate(data['evidences'], start=1):
        ev_text = get_evidence_text(ev_id)
        if ev_text:
            evidence_pieces.append(f"{i}. {ev_text}")
        else:
            print(f"Warning: {ev_id} not found in DB, skipping.")

    if not evidence_pieces:
        print(f"Skipping {cid}: no evidence retrieved.")
        continue

    evidence_text = " ".join(evidence_pieces)
    claim_text = data['claim_text']
    label = data['claim_label']

    prompt = format_prompt(evidence_text, claim_text, label)
    all_claims_ready.append({
        "cid": cid,
        "prompt": prompt,
        "claim_text": claim_text,
        "label": label
    })

print(f"To generate: {len(all_claims_ready)} claims (batch_size={BATCH_SIZE})")

for batch_start in tqdm(range(0, len(all_claims_ready), BATCH_SIZE), desc="Generating CoT"):
    batch_items = all_claims_ready[batch_start: batch_start + BATCH_SIZE]

    reasonings = generate_batch(batch_items, max_new_tokens=MAX_NEW_TOKENS)

    for item, reasoning in zip(batch_items, reasonings):
        cursor.execute(
            "INSERT OR REPLACE INTO reasoning_data VALUES (?, ?, ?, ?)",
            (item["cid"], item["claim_text"], item["label"], reasoning)
        )

    # Commit after every batch so progress is always saved
    db_conn.commit()

db_conn.close()
print("Done! All CoT reasoning saved to DB.")

## 6.LoRA Fine-Tuning
* Loads data from the CoT database and converts it into the ChatML format (System, User, Assistant roles).
* Configures LoRA parameters (Rank=16, Alpha=32) using the peft library to train only a small subset of model weights.
* Uses TRL's SFTTrainer to execute the training process and saves the resulting model adapters.

In [ ]:
import pathlib
import os

os.environ["PYTHONUTF8"] = "1"
_orig_read_text = pathlib.Path.read_text
def _patched_read_text(self, encoding=None, errors=None):
    if encoding is None:
        encoding = "utf-8"
    return _orig_read_text(self, encoding=encoding, errors=errors)
pathlib.Path.read_text = _patched_read_text
# ─────────────────────────────────────────────────────────────────────────────

import sqlite3
import pandas as pd
import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Loading data from database...")
conn = sqlite3.connect(COT_DB_PATH)

query = "SELECT * FROM reasoning_data"
df = pd.read_sql_query(query, conn)
conn.close()

print("Formatting data into ChatML...")

def format_chatml(row):

    return {
        "messages": [
            {"role": "system", "content": "You are a factual verification assistant. Think step-by-step to classify the claim based on the evidence."},
            {"role": "user", "content": str(row['claim_text'])},
            {"role": "assistant", "content": str(row['reasoning'])}
        ]
    }

chat_data = df.apply(format_chatml, axis=1).tolist()
train_dataset = Dataset.from_list(chat_data)


print("Loading Model and Tokenizer in 4-bit...")
model_id = "Qwen/Qwen3.5-2B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

print("Applying LoRA...")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("Starting Training...")
training_args = SFTConfig(
    output_dir="model/qwen-cot-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    save_strategy="epoch",
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_length=1024
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"Training complete! LoRA weights saved to {LORA_PATH}")

## 7. Model Evaluation and Comparison
Compares the accuracy of the original base model (using few-shot prompting) against the fine-tuned model on a development set.

In [ ]:
import os
import json
import re
import torch
from tqdm import tqdm
import gc

def extract_label(text):
    text_upper = text.upper()
    matches = re.finditer(r'\b(SUPPORTS|REFUTES|NOT_ENOUGH_INFO|DISPUTED)\b', text_upper)
    found_labels = [m.group(0) for m in matches]

    if found_labels:
        return found_labels[-1]
    else:
        return "UNKNOWN_FORMAT"

def evaluate_model(verifier, data_items, few_shot=False, batch_size=4):
    correct_count = 0
    total_count = len(data_items)

    for i in tqdm(range(0, total_count, batch_size), desc="Evaluating Batches"):
        batch_items = data_items[i:i + batch_size]

        batch_claim_texts = []
        batch_true_labels = []
        batch_evidences = []

        for claim_id, claim_info in batch_items:
            batch_claim_texts.append(claim_info.get('claim_text', ''))
            batch_true_labels.append(claim_info.get('claim_label', ''))
            batch_evidences.append(claim_info.get('evidences', []))

        batch_model_outputs = verifier.predict(
            batch_claim_texts,
            batch_evidence_ids=batch_evidences,
            few_shot=few_shot
        )

        for true_label, output in zip(batch_true_labels, batch_model_outputs):
            predicted_label = extract_label(output)
            if predicted_label == true_label:
                correct_count += 1

    accuracy = correct_count / total_count if total_count > 0 else 0
    return accuracy, correct_count, total_count

def compare_models():
    BATCH_SIZE = 4
    NUM_SAMPLES = 50

    print(f"Loading top {NUM_SAMPLES} claims from {DEV_FILE_PATH}...")
    with open(DEV_FILE_PATH, 'r', encoding='utf-8') as f:
        dev_data = json.load(f)

    dev_items = list(dev_data.items())[:NUM_SAMPLES]
    actual_samples = len(dev_items)

    print("\n" + "="*50)
    print("STEP 1: Evaluating Base Model (with Few-Shot)")
    print("="*50)
    base_verifier = ClaimVerifier(base_model_id=MODEL_ID, lora_path=None)
    base_acc, base_correct, _ = evaluate_model(base_verifier, dev_items, few_shot=True, batch_size=BATCH_SIZE)

    del base_verifier
    gc.collect()
    torch.cuda.empty_cache()
    print("Base Model memory cleared.")

    print("\n" + "="*50)
    print("STEP 2: Evaluating Fine-Tuned Model (Zero-Shot / Direct)")
    print("="*50)
    ft_verifier = ClaimVerifier(base_model_id=MODEL_ID, lora_path=LORA_PATH)
    ft_acc, ft_correct, _ = evaluate_model(ft_verifier, dev_items, few_shot=False, batch_size=BATCH_SIZE)

    del ft_verifier
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Base Model Accuracy      : {base_acc:.2%} ({base_correct}/{actual_samples})")
    print(f"Fine-Tuned Model Accuracy: {ft_acc:.2%} ({ft_correct}/{actual_samples})")

if __name__ == "__main__":
    compare_models()

## 8. Constructing the Vector Database
Creates a searchable vector index for all evidence text to enable semantic (RAG) retrieval.

In [ ]:
import sqlite3
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

BATCH_SIZE = 256

model = SentenceTransformer(RAG_MODEL_NAME)
conn = sqlite3.connect(EVIDENCE_DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT id, text FROM evidence")
rows = cursor.fetchall()
ids = [row[0] for row in rows]
texts = [row[1] for row in rows]
print(f"Loaded {len(texts)} data, start to vectorize...")
embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_embeddings = model.encode(batch_texts, convert_to_numpy=True, normalize_embeddings=True)
    embeddings.append(batch_embeddings)
embeddings = np.vstack(embeddings)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, FAISS_INDEX_PATH)
with open(FAISS_META_PATH, 'w', encoding='utf-8') as f:
    json.dump({'ids': ids}, f)
print("Vectorize complete!")
conn.close()

## 9. RAG Pipeline Implementation
**Retrieval:** Uses a Bi-Encoder to find the top-20 most similar evidence candidates from the FAISS index.

**Re-ranking:** Uses a CrossEncoder to calculate more accurate relevancy scores between the claim and the candidates.

**Filtering:** Keeps only documents with a relevancy score higher than 2.5.

In [ ]:
import sqlite3
import json
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder

BI_ENCODER_NAME = 'all-MiniLM-L6-v2'
CROSS_ENCODER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

class RAGPipeline:
    def __init__(self):
        print("Initializing RAG Pipeline...")
        self.bi_encoder = SentenceTransformer(BI_ENCODER_NAME)
        self.cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)
        self.index = faiss.read_index(FAISS_INDEX_PATH)
        with open(FAISS_META_PATH, 'r', encoding='utf-8') as f:
            self.metadata = json.load(f)
        self.ids = self.metadata['ids']

        self.conn = sqlite3.connect(EVIDENCE_DB_PATH)

    def get_text_by_id(self, id):
        cursor = self.conn.cursor()
        cursor.execute("SELECT text FROM evidence WHERE id = ?", (id,))
        result = cursor.fetchone()
        return result[0] if result else ""

    def process_claim(self, claim_text, top_k_retrieve=20, threshold=2.5):
        claim_embedding = self.bi_encoder.encode([claim_text], convert_to_numpy=True, normalize_embeddings=True)
        distances, indices = self.index.search(claim_embedding, top_k_retrieve)

        retrieved_docs = []
        for idx in indices[0]:
            doc_id = self.ids[idx]
            doc_text = self.get_text_by_id(doc_id)
            retrieved_docs.append({
                "id": doc_id,
                "text": doc_text
            })

        cross_inp = [[claim_text, doc["text"]] for doc in retrieved_docs]
        cross_scores = self.cross_encoder.predict(cross_inp)

        for i in range(len(retrieved_docs)):
            retrieved_docs[i]["rerank_score"] = float(cross_scores[i])

        reranked_docs = sorted(retrieved_docs, key=lambda x: x["rerank_score"], reverse=True)

        final_docs = [doc for doc in reranked_docs if doc["rerank_score"] >= threshold]

        if not final_docs and reranked_docs:
            final_docs = [reranked_docs[0]]

        return final_docs

## 10. Claim Verifier Class
Encapsulates the entire inference logic, including model loading, evidence fetching, and prediction.

In [ ]:
import os
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

class ClaimVerifier:
    def __init__(self, base_model_id="Qwen/Qwen3.5-2B", lora_path=None):
        print("Loading Tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print("Configuring 4-bit Quantization...")
        compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype
        )

        print(f"Loading Base Model: {base_model_id}...")
        self.model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=compute_dtype,
            attn_implementation="sdpa"
        )

        if lora_path and os.path.exists(lora_path):
            print(f"Applying LoRA weights from {lora_path}...")
            self.model = PeftModel.from_pretrained(self.model, lora_path)
            print("LoRA loaded successfully!")
        else:
            print("No valid LoRA path provided. Using Base Model for inference.")

        self.rag_pipeline = None

    def fetch_evidence(self, claim_text, evidence_ids=None):
        evidence_texts = []

        if evidence_ids and len(evidence_ids) > 0:
            print(f"Using provided evidence IDs: {evidence_ids}")
            for eid in evidence_ids:
                text = get_evidence_text(eid)
                if text:
                    evidence_texts.append(f"[{eid}] {text}")
        else:
            print("No evidence IDs provided. Triggering RAG Pipeline...")
            if self.rag_pipeline is None:
                self.rag_pipeline = RAGPipeline()

            top_evidence = self.rag_pipeline.process_claim(claim_text, top_k_retrieve=20)
            for ev in top_evidence:
                evidence_texts.append(f"[{ev['id']}] {ev['text']}")

        return "\n".join(evidence_texts)

    def predict(self, claims_texts, batch_evidence_ids, few_shot=False):
        self.tokenizer.padding_side = 'left'
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        text_prompts = []

        for claim_text, evidence_ids in zip(claims_texts, batch_evidence_ids):
            combined_evidence = self.fetch_evidence(claim_text, evidence_ids)

            if few_shot:
                messages = [
                    {"role": "system", "content": """You are a climate science fact-checker. Your task is to explain the logical connection between the Evidence and the Claim to justify the Label. Use the format: "Let's analyze step by step: [Reasoning] Therefore, the conclusion is: [Label]." conclude your response with one of the following four labels:
                    [SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED]. Here are some examples:
                    Evidence: 1. CO2 can be toxic to animals at 10,000 ppm. 2. Plants grow faster at 1,000 ppm CO2. 3. Higher CO2 affects plant growth favorably.
                    Claim: Higher CO2 concentrations actually help ecosystems support more plant and animal life.
                    Label: DISPUTED
                    Let's analyze step by step: The evidence confirms that CO2 promotes plant growth, which supports part of the claim. However, it also notes that extremely high concentrations are toxic to animal life. Since the claim makes a broad positive statement without accounting for these toxic thresholds, the claim is partially accurate but also potentially dangerous/misleading.
                    Therefore, the conclusion is: DISPUTED.

                    Evidence: 1. Human activity and GHG emissions are key factors in global temperature increases. 2. Warming is driven by human-caused thermal expansion and melting ice.
                    Claim: El Niño drove record highs in global temperatures suggesting rise may not be down to man-made emissions.
                    Label: REFUTES
                    Let's analyze step by step: While El Niño is a natural driver of temperature, the evidence explicitly states that human activity is the "key factor" in the pace of current temperature increases. The claim attempts to dismiss man-made emissions by pointing to a natural cause, which contradicts the "substantial evidence" mentioned in the text regarding human-caused warming.
                    Therefore, the conclusion is: REFUTES.

                    Evidence: 1. Reversals in polarity occurred around 1925, 1947, and 1977. 2. The PDO changed to a "cool" phase in a regime shift similar to the 1970s.
                    Claim: In 1946, PDO switched to a cool phase.
                    Label: SUPPORTS
                    Let's analyze step by step: The evidence mentions a major PDO reversal occurring around 1947 and explicitly describes a shift to a "cool" phase. The year 1946 is immediately adjacent to the 1947 reversal date cited. Given the context of regime shifts, the evidence provides sufficient support for the timing and nature of the phase change described in the claim.
                    Therefore, the conclusion is: SUPPORTS."""},
                    {"role": "user", "content": f"Evidence:\n{combined_evidence}\n\nClaim:\n{claim_text}"}
                ]
            else:
                messages = [
                    {"role": "system", "content": "You are a climate science fact-checker. Your task is to explain the logical connection between the Evidence and the Claim to justify the Label."},
                    {"role": "user", "content": f"Evidence:\n{combined_evidence}\n\nClaim:\n{claim_text}"}
                ]

            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            text_prompts.append(prompt)

        inputs = self.tokenizer(
            text_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(self.model.device)

        print(f"Generating predictions for a batch of {len(claims_texts)} claims...")

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.1,
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id
            )

        input_length = inputs.input_ids.shape[1]
        clean_responses = []

        for output in outputs:
            generated_tokens = output[input_length:]
            full_response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

            clean_response = full_response
            for word in ["Evidence:", "Claim:", "Label:"]:
                if word in clean_response:
                    clean_response = clean_response.split(word)[0]

            clean_responses.append(clean_response.strip())

        return clean_responses

## 11. Final Test Set Prediction
Instantiates the ClaimVerifier, loads the test claims, and processes them in batches (Batch Size: 6) to produce the final output.

In [ ]:
import os
import sys
import json
import re
from tqdm import tqdm

def extract_label(text):
    text_upper = text.upper()
    matches = re.finditer(r'\b(SUPPORTS|REFUTES|NOT_ENOUGH_INFO|DISPUTED)\b', text_upper)
    found_labels = [m.group(0) for m in matches]

    if found_labels:
        return found_labels[-1]
    else:
        return "NOT_ENOUGH_INFO"

def generate_test_predictions_batched(base_model_id, lora_path, test_file_path, output_filepath, batch_size=3):
    print("Initializing ClaimVerifier (Model & LoRA)...")
    verifier = ClaimVerifier(base_model_id=base_model_id, lora_path=lora_path)

    print("Initializing RAG Pipeline (Retriever & Re-ranker)...")
    rag = RAGPipeline()

    print(f"Loading Unlabelled Test Set from {test_file_path}...")
    with open(test_file_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)

    output_data = {}

    items = list(test_data.items())
    total_claims = len(items)

    print(f"Starting Prediction on {total_claims} claims (Batch Size: {batch_size})...")

    for i in tqdm(range(0, total_claims, batch_size), desc="Predicting Batches", colour="green"):
        batch_items = items[i:i + batch_size]

        batch_claim_ids = []
        batch_claim_texts = []
        batch_evidence_ids = []

        for claim_id, claim_info in batch_items:
            claim_text = claim_info.get('claim_text', '')

            top_evidence = rag.process_claim(claim_text, top_k_retrieve=20)
            evidence_ids = [ev['id'] for ev in top_evidence]

            batch_claim_ids.append(claim_id)
            batch_claim_texts.append(claim_text)
            batch_evidence_ids.append(evidence_ids)

        model_outputs = verifier.predict(batch_claim_texts, batch_evidence_ids, few_shot=False)


        for claim_id, claim_text, ev_ids, model_output in zip(batch_claim_ids, batch_claim_texts, batch_evidence_ids, model_outputs):
            predicted_label = extract_label(model_output)

            output_data[claim_id] = {
                "claim_text": claim_text,
                "claim_label": predicted_label,
                "evidences": ev_ids
            }

    print(f"\nSaving predictions to {output_filepath}...")

    output_dir = os.path.dirname(output_filepath)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"✅ All Done! Results saved to {output_filepath}.")

if __name__ == "__main__":
    TEST_FILE = "test-claims-unlabelled.json"
    OUTPUT_FILE = "test-output.json"

    generate_test_predictions_batched(
        base_model_id=MODEL_ID,
        lora_path=LORA_PATH,
        test_file_path=TEST_FILE,
        output_filepath=OUTPUT_FILE,
        batch_size=6
    )